In [ ]:
# READ IN HAIL (need these paths to prevent errors)
import hail as hl
import os

hail_jar = os.path.join(os.path.dirname(hl.__file__), "backend", "hail-all-spark.jar")

hl.init(
    spark_conf={
        "spark.jars": hail_jar,
        "spark.driver.extraClassPath": hail_jar,
        "spark.executor.extraClassPath": hail_jar,
    }
)

## Counts of other ACs
Ran this changin the upper and lower filter in bins of up to AC50. 

In [ ]:
# 6 TO 10 AC COUNTS 
# Define the chromosomes you are working with
chromosomes = list(range(1, 23))
AC_filter_lower=6
AC_filter_upper=10

for chr in chromosomes:
    if chr == 1 :
        # First half 
        print(f"Processing chromosome {chr} first half...")
        ####### PREP MT ######### 
        # Only do this the first time you're running counts - not needed at later stages as the count ready matrix table is written out, so can be read in for later counts. 
         # Read the matrix table for the current chromosome
        mt=hl.read_matrix_table("dnax://database-Gq45XQjJ637Q9X6XJJJ3Pf7k/chr_1_first_half_ready_for_counts.mt")
        print(f'AC filter set as {AC_filter_lower} - {AC_filter_upper}')
        filtered_mt=mt.filter_rows((mt.variant_qc.AC[1]<= AC_filter_upper) & (mt.variant_qc.AC[1]>= AC_filter_lower))
        filtered_mt.checkpoint(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_first_half.mt', overwrite=True)
        ####### COUNTS ########
        # PTVs
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_first_half.mt')
        PTV_var = (filtered_mt
           .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
           .aggregate(
               n= hl.agg.filter(
                   (filtered_mt.LoF_worstCsq==True)& 
                   (filtered_mt.GT.is_non_ref()),
                   hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} first half done...')
        PTV_var.n.export(f"chr_{chr}_first_half_PTV_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} first half written out')
        print('Dont forget to copy these tables up to your project before closing the session!!')
        # Deleterious missense variants 
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_first_half.mt') # Read back in here as it speeds up process. Checkpoint command is suposed to read back in but doesn't seem to be?
        Revel75_miss_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Miss_worstCsq == True)& 
                     (filtered_mt.REVEL_score> 0.75),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} first half done...')
        Revel75_miss_var.n.export(f"chr_{chr}_first_half_REVEL75_Miss_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} first half written out')
        print('Dont forget to copy these tables up to your project before closing the session!!')
        # Synonymous variants 
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_first_half.mt')
        syn_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Syn_worstCsq == True),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} first half done...')
        syn_var.n.export(f"chr_{chr}_first_half_synonymous_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for {chr} first half written out')
        print('Dont forget to copy these tables up to your project before closing the session!!')
        print(f"Finished processing chromosome {chr} first half!")
    
    
        # Second half
        print(f"Processing chromosome {chr} second half...")
        ####### PREP MT ######### 
        # Read the matrix table for the current chromosome
        mt=hl.read_matrix_table("dnax://database-Gq45XQjJ637Q9X6XJJJ3Pf7k/chr_1_second_half_ready_for_counts.mt")
        print(f'AC filter set as {AC_filter_lower} - {AC_filter_upper}')
        filtered_mt=mt.filter_rows((mt.variant_qc.AC[1]<= AC_filter_upper) & (mt.variant_qc.AC[1]>= AC_filter_lower))
        filtered_mt.checkpoint(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_second_half.mt', overwrite=True)
        ###### COUNTS #######
        # PTVs 
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_second_half.mt')
        PTV_var = (filtered_mt
           .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
           .aggregate(
               n= hl.agg.filter(
                   (filtered_mt.LoF_worstCsq==True)& 
                   (filtered_mt.GT.is_non_ref()),
                   hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} second half done...')
        PTV_var.n.export(f"chr_{chr}_second_half_PTV_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'PTV AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} second half written out')
        # Deleterious missense variants
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_second_half.mt') # Read back in here as it speeds up process. Checkpoint command is suposed to read back in but doesn't seem to be?
        Revel75_miss_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Miss_worstCsq == True)& 
                     (filtered_mt.REVEL_score> 0.75),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} second half done...')
        Revel75_miss_var.n.export(f"chr_{chr}_second_half_REVEL75_Miss_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} second half written out')
        # Count synonymous variants and write out
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}_second_half.mt')
        syn_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Syn_worstCsq == True),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} second half done...')
        syn_var.n.export(f"chr_{chr}_second_half_synonymous_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for {chr} second half written out')
        print('Dont forget to copy these tables up to your project before closing the session!!')
        print(f"Finished processing chromosome {chr} second half!")
        print(f"Finished processing chromosome {chr}!")

    
    else:
        print(f"Processing chromosome {chr}...")
        ######## PREP MT #########
        # Read the matrix table for the current chromosome
        mt=hl.read_matrix_table(f'dnax://database-Gq45XQjJ637Q9X6XJJJ3Pf7k/chr_{chr}_ready_for_counts.mt')
        print(f'AC filter set as {AC_filter_lower} - {AC_filter_upper}')
        filtered_mt=mt.filter_rows((mt.variant_qc.AC[1]<= AC_filter_upper) & (mt.variant_qc.AC[1]>= AC_filter_lower))
        filtered_mt.checkpoint(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}.mt', overwrite=True)
        ####### COUNTS ####### 
        # PTVs
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}.mt')
        PTV_var = (filtered_mt
           .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
           .aggregate(
               n= hl.agg.filter(
                   (filtered_mt.LoF_worstCsq==True)& 
                   (filtered_mt.GT.is_non_ref()),
                   hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} done...')
        PTV_var.n.export(f"chr_{chr}_PTV_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'PTV AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} written out')
        # Deleterious missense variants and write out
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}.mt') # Read back in here as it speeds up process. Checkpoint command is suposed to read back in but doesn't seem to be?
        Revel75_miss_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Miss_worstCsq == True)& 
                     (filtered_mt.REVEL_score> 0.75),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} done...')
        Revel75_miss_var.n.export(f"chr_{chr}_REVEL75_Miss_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'REVEL >0.75 missense AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} written out')
        # Count synonymous variants and write out
        #filtered_mt=hl.read_matrix_table(f'AC{AC_filter_lower}to{AC_filter_upper}_chr_{chr}.mt')
        syn_var=(filtered_mt
             .group_rows_by(gene_id=filtered_mt.gene_id_worstCsq)
             .aggregate(
                 n = hl.agg.filter(
                     (filtered_mt.Syn_worstCsq == True),
                     hl.agg.sum(filtered_mt.GT.n_alt_alleles()))))
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for chr {chr} done...')
        syn_var.n.export(f"chr_{chr}_synonymous_AC{AC_filter_lower}to{AC_filter_upper}_gene_counts.tsv")
        print(f'Synonymous AC{AC_filter_lower} - {AC_filter_upper} counts for {chr} written out')
        print('Dont forget to copy these tables up to your project before closing the session!!')
        print(f"Finished processing chromosome {chr}!")
    